<a href="https://colab.research.google.com/github/theboogeyman81/for_deep_learning/blob/main/RPS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import pickle

In [3]:
import os

DATASET_DIR = '/content/drive/MyDrive/rock'


In [7]:
import os
import shutil
from sklearn.model_selection import train_test_split

# Define the new base directories for organized data
BASE_DIR = '/content/rps_dataset'
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
TEST_DIR = os.path.join(BASE_DIR, 'test')

# Define the classes based on your current directory structure
CLASS_NAMES = [d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d)) and d in ['rock', 'paper', 'scissors']]

# Create base directories if they don't exist
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

print(f"Creating new dataset structure in: {BASE_DIR}")
print(f"Classes found: {CLASS_NAMES}")

# Create class subdirectories within train and test
for class_name in CLASS_NAMES:
    os.makedirs(os.path.join(TRAIN_DIR, class_name), exist_ok=True)
    os.makedirs(os.path.join(TEST_DIR, class_name), exist_ok=True)

print("Directory structure created.")

Creating new dataset structure in: /content/rps_dataset
Classes found: ['paper', 'rock', 'scissors']
Directory structure created.


In [8]:
import os
import shutil
from sklearn.model_selection import train_test_split

# Redefine TRAIN_DIR and TEST_DIR to point to the newly created structure
TRAIN_DIR = '/content/rps_dataset/train'
TEST_DIR = '/content/rps_dataset/test'

# Now, let's split the data
for class_name in CLASS_NAMES:
    class_path = os.path.join(DATASET_DIR, class_name)
    images = [os.path.join(class_path, img) for img in os.listdir(class_path) if img.endswith(('.jpg', '.jpeg', '.png'))]

    # Split data (80% train, 20% test)
    train_images, test_images = train_test_split(images, test_size=0.2, random_state=42)

    print(f"Processing class: {class_name}")
    print(f"  Total images: {len(images)}")
    print(f"  Training images: {len(train_images)}")
    print(f"  Test images: {len(test_images)}")

    # Move training images
    for img_path in train_images:
        shutil.copy(img_path, os.path.join(TRAIN_DIR, class_name, os.path.basename(img_path)))

    # Move test images
    for img_path in test_images:
        shutil.copy(img_path, os.path.join(TEST_DIR, class_name, os.path.basename(img_path)))

print("Image splitting and copying complete.")

# Verify the new paths
print(f"\nUpdated TRAIN_DIR to: {TRAIN_DIR}")
print(f"Updated TEST_DIR to: {TEST_DIR}")


Processing class: paper
  Total images: 713
  Training images: 570
  Test images: 143
Processing class: rock
  Total images: 726
  Training images: 580
  Test images: 146
Processing class: scissors
  Total images: 750
  Training images: 600
  Test images: 150
Image splitting and copying complete.

Updated TRAIN_DIR to: /content/rps_dataset/train
Updated TEST_DIR to: /content/rps_dataset/test


In [10]:
IMG_SIZE = 150
BATCH_SIZE = 32
EPOCHS = 25

print(f"Training folder: {TRAIN_DIR}")
print(f"Test folder: {TEST_DIR}")
print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")

Training folder: /content/rps_dataset/train
Test folder: /content/rps_dataset/test
Image size: 150x150


In [14]:
TRAIN_DIR = '/content/rps_dataset/train'
TEST_DIR = '/content/rps_dataset/test'

In [12]:
IMG_SIZE = 150
BATCH_SIZE = 32
EPOCHS = 25

print(f"Training folder: {TRAIN_DIR}")
print(f"Test folder: {TEST_DIR}")
print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")

Training folder: /content/rps
Test folder: /content/rps-test-set
Image size: 150x150


In [15]:
print("\n Loading data...")

# Training data with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

# Test data (only rescale)
test_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_data = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"Training images: {train_data.samples}")
print(f"Test images: {test_data.samples}")
print(f"Classes: {list(train_data.class_indices.keys())}")


 Loading data...
Found 1750 images belonging to 3 classes.
Found 439 images belonging to 3 classes.
Training images: 1750
Test images: 439
Classes: ['paper', 'rock', 'scissors']


In [16]:
print("\nBuilding model...")

model = Sequential([
    # Block 1
    Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    MaxPooling2D(2, 2),

    # Block 2
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    # Block 3
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    # Block 4
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    # Dense layers
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')  # 3 classes: rock, paper, scissors
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


Building model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 15, 15, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     3,211,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │         1,539 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,454,147 (13.18 MB)

 Trainable params: 3,454,147 (13.18 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
print("\n Loading data...")

# Training data with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

# Test data (only rescale)
test_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_data = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"Training images: {train_data.samples}")
print(f"Test images: {test_data.samples}")
print(f"Classes: {list(train_data.class_indices.keys())}")


 Loading data...
Found 1750 images belonging to 3 classes.
Found 439 images belonging to 3 classes.
Training images: 1750
Test images: 439
Classes: ['paper', 'rock', 'scissors']


In [18]:
print("\nTraining model...")
print("This will take 8-10 minutes with GPU...\n")

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    ModelCheckpoint('best_rps_model.h5', monitor='val_accuracy', save_best_only=True)
]

history = model.fit(
    train_data,
    epochs=EPOCHS,
    validation_data=test_data,
    callbacks=callbacks
)

print("\nTraining complete!")


Training model...
This will take 8-10 minutes with GPU...



/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.3714 - loss: 1.0902

55/55 ━━━━━━━━━━━━━━━━━━━━ 75s 1s/step - accuracy: 0.3723 - loss: 1.0894 - val_accuracy: 0.7927 - val_loss: 0.6105
Epoch 2/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6867 - loss: 0.7173

55/55 ━━━━━━━━━━━━━━━━━━━━ 67s 1s/step - accuracy: 0.6872 - loss: 0.7164 - val_accuracy: 0.8519 - val_loss: 0.3957
Epoch 3/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7510 - loss: 0.5998

55/55 ━━━━━━━━━━━━━━━━━━━━ 65s 1s/step - accuracy: 0.7513 - loss: 0.5991 - val_accuracy: 0.9362 - val_loss: 0.2382
Epoch 4/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7942 - loss: 0.4952

55/55 ━━━━━━━━━━━━━━━━━━━━ 66s 1s/step - accuracy: 0.7946 - loss: 0.4946 - val_accuracy: 0.9681 - val_loss: 0.2072
Epoch 5/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8642 - loss: 0.3749

55/55 ━━━━━━━━━━━━━━━━━━━━ 66s 1s/step - accuracy: 0.8644 - loss: 0.3744 - val_accuracy: 0.9909 - val_loss: 0.0909
Epoch 6/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 65s 1s/step - accuracy: 0.9126 - loss: 0.2643 - val_accuracy: 0.9863 - val_loss: 0.0538
Epoch 7/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9509 - loss: 0.1892

55/55 ━━━━━━━━━━━━━━━━━━━━ 64s 1s/step - accuracy: 0.9507 - loss: 0.1893 - val_accuracy: 0.9932 - val_loss: 0.0434
Epoch 8/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 65s 1s/step - accuracy: 0.9564 - loss: 0.1376 - val_accuracy: 0.9727 - val_loss: 0.0734
Epoch 9/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 64s 1s/step - accuracy: 0.9625 - loss: 0.1141 - val_accuracy: 0.9658 - val_loss: 0.1038
Epoch 10/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 71s 1s/step - accuracy: 0.9500 - loss: 0.1521 - val_accuracy: 0.9886 - val_loss: 0.0398
Epoch 11/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 68s 1s/step - accuracy: 0.9500 - loss: 0.1425 - val_accuracy: 0.9886 - val_loss: 0.0224
Epoch 12/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 63s 1s/step - accuracy: 0.9640 - loss: 0.1104 - val_accuracy: 0.9727 - val_loss: 0.0888

Training complete!


In [19]:
print("\n Evaluating model...")

test_data.reset()
loss, accuracy = model.evaluate(test_data)



 Evaluating model...
14/14 ━━━━━━━━━━━━━━━━━━━━ 4s 288ms/step - accuracy: 0.9887 - loss: 0.0570


In [20]:
test_data.reset()
predictions = model.predict(test_data)
y_pred = np.argmax(predictions, axis=1)
y_true = test_data.classes

14/14 ━━━━━━━━━━━━━━━━━━━━ 5s 351ms/step


In [21]:
class_names = list(train_data.class_indices.keys())
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))


Classification Report:
              precision    recall  f1-score   support

       paper       0.99      0.99      0.99       143
        rock       1.00      0.99      1.00       146
    scissors       0.99      1.00      0.99       150

    accuracy                           0.99       439
   macro avg       0.99      0.99      0.99       439
weighted avg       0.99      0.99      0.99       439



In [22]:
print("\n Creating confusion matrix...")

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
print("✅ Saved confusion_matrix.png")
plt.close()


 Creating confusion matrix...
✅ Saved confusion_matrix.png


In [23]:
print("\nSaving model...")

model.save('rps_model.h5')
print("✅ Saved rps_model.h5")

# Save metadata
metadata = {
    'image_size': IMG_SIZE,
    'classes': class_names,
    'accuracy': float(accuracy),
    'loss': float(loss)
}

with open('metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)
print("✅ Saved metadata.pkl")




Saving model...
✅ Saved rps_model.h5
✅ Saved metadata.pkl


In [24]:
from google.colab import files

files.download('rps_model.h5')
files.download('metadata.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>